In [1]:
"""
=============================================================================
 Exercise 3 : Feedforward Neural Network Diagnostic System
              Detecting Learning-Rate, Gradient and Overfitting Failures
              Dataset : Fashion-MNIST (Zalando Research)
=============================================================================
 Models used
   FNN  : single hidden layer feedforward network
   MLP  : 2-3 hidden layer multilayer perceptron
   DFNN : deep feedforward network (6+ hidden layers)

 The system TRAINS the networks and then DIAGNOSES them from real measured
 quantities:
   * epoch-wise training / validation loss and accuracy
   * per-layer gradient magnitudes  (captured with tf.GradientTape)
   * per-layer neuron activation outputs (captured with a probe sub-model)

 Requirements:  tensorflow>=2.10   numpy   matplotlib
 Run:           python nn_failure_diagnosis.py
=============================================================================
"""

import os
import warnings

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
warnings.filterwarnings("ignore")

import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    PLOTTING = True
except Exception:
    PLOTTING = False

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

CLASS_NAMES = ["T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
               "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]

# Results of every experiment that has been run (used by menu option 7)
RESULTS = {}


# =============================================================================
# 1. DATA
# =============================================================================
def load_data(train_subset=12000, test_subset=3000, tiny_train=None):
    """
    Loads Fashion-MNIST, flattens 28x28 -> 784 and scales to [0,1].

    train_subset : sub-sample size used for normal experiments (keeps the
                   menu responsive; set to None to use all 60,000 images)
    tiny_train   : if given, an intentionally small training set used for
                   the OVERFITTING experiment.
    """
    (x_tr, y_tr), (x_te, y_te) = keras.datasets.fashion_mnist.load_data()

    x_tr = x_tr.reshape(-1, 784).astype("float32") / 255.0
    x_te = x_te.reshape(-1, 784).astype("float32") / 255.0

    n_train = tiny_train if tiny_train else train_subset
    if n_train:
        x_tr, y_tr = x_tr[:n_train], y_tr[:n_train]
    if test_subset:
        x_te, y_te = x_te[:test_subset], y_te[:test_subset]

    return (x_tr, y_tr), (x_te, y_te)


# =============================================================================
# 2. MODEL BUILDERS  (FNN / MLP / DFNN)
# =============================================================================
def build_model(arch="MLP", activation="relu", init="glorot_uniform",
                dropout=0.0, l2=0.0, width=128):
    """Builds FNN, MLP or DFNN with a chosen activation function."""
    reg = keras.regularizers.l2(l2) if l2 > 0 else None

    if arch == "FNN":
        hidden = [width]
    elif arch == "MLP":
        hidden = [width, width // 2]
    elif arch == "DFNN":
        hidden = [width] * 6            # deep -> exposes vanishing gradients
    else:
        raise ValueError("arch must be FNN, MLP or DFNN")

    model = keras.Sequential(name=f"{arch}_{activation}")
    model.add(layers.Input(shape=(784,)))
    for i, units in enumerate(hidden):
        model.add(layers.Dense(units, activation=activation,
                               kernel_initializer=init,
                               kernel_regularizer=reg,
                               name=f"dense_{i+1}"))
        if dropout > 0:
            model.add(layers.Dropout(dropout, name=f"drop_{i+1}"))
    model.add(layers.Dense(10, activation="softmax", name="output"))
    return model


# =============================================================================
# 3. INSTRUMENTATION
# =============================================================================
LOSS_FN = keras.losses.SparseCategoricalCrossentropy()


class GradientMonitor(keras.callbacks.Callback):
    """Records the mean |gradient| of every Dense kernel after each epoch."""

    def __init__(self, x_probe, y_probe):
        super().__init__()
        self.x = tf.convert_to_tensor(x_probe)
        self.y = tf.convert_to_tensor(y_probe)
        self.history = []          # list of dict{layer_name: mean_abs_grad}

    def on_epoch_end(self, epoch, logs=None):
        with tf.GradientTape() as tape:
            preds = self.model(self.x, training=True)
            loss = LOSS_FN(self.y, preds)
        weights = [w for w in self.model.trainable_weights if "kernel" in w.name]
        grads = tape.gradient(loss, weights)
        rec = {}
        for w, g in zip(weights, grads):
            name = w.name.split("/")[0]
            rec[name] = float(tf.reduce_mean(tf.abs(g))) if g is not None else 0.0
        self.history.append(rec)

    def final(self):
        return self.history[-1] if self.history else {}


def measure_activations(model, x_probe):
    """
    Runs a probe batch through the network and returns, for every hidden
    Dense layer, the fraction of neurons that are dead (output == 0 for
    EVERY sample in the probe batch) plus the mean activation.
    """
    hidden = [l for l in model.layers
              if isinstance(l, layers.Dense) and l.name != "output"]
    if not hidden:
        return {}
    probe = keras.Model(model.inputs, [l.output for l in hidden])
    outs = probe.predict(x_probe, verbose=0)
    if not isinstance(outs, list):
        outs = [outs]

    stats = {}
    for layer, act in zip(hidden, outs):
        active_per_neuron = (act > 1e-8).sum(axis=0)          # per neuron
        dead = float((active_per_neuron == 0).mean())
        stats[layer.name] = {"dead_fraction": dead,
                             "mean_activation": float(act.mean()),
                             "units": act.shape[1]}
    return stats


def train(model, data, lr, epochs, optimizer="sgd", batch_size=128,
          verbose=0, label=""):
    """Compiles + trains a model and returns a bundle of measured metrics."""
    (x_tr, y_tr), (x_te, y_te) = data

    opt = (keras.optimizers.Adam(learning_rate=lr) if optimizer == "adam"
           else keras.optimizers.SGD(learning_rate=lr))
    model.compile(optimizer=opt, loss="sparse_categorical_crossentropy",
                  metrics=["accuracy"])

    probe_x, probe_y = x_tr[:512], y_tr[:512]
    gmon = GradientMonitor(probe_x, probe_y)

    print(f"\n  >> Training {model.name} | optimizer={optimizer} "
          f"| lr={lr} | epochs={epochs}")
    hist = model.fit(x_tr, y_tr,
                     validation_data=(x_te, y_te),
                     epochs=epochs, batch_size=batch_size,
                     callbacks=[gmon], verbose=verbose)

    acts = measure_activations(model, probe_x)

    bundle = {
        "label": label or model.name,
        "model": model.name,
        "lr": lr,
        "optimizer": optimizer,
        "epochs": epochs,
        "train_loss": [float(v) for v in hist.history["loss"]],
        "val_loss": [float(v) for v in hist.history["val_loss"]],
        "train_acc": [float(v) for v in hist.history["accuracy"]],
        "val_acc": [float(v) for v in hist.history["val_accuracy"]],
        "grad_history": gmon.history,
        "grad_final": gmon.final(),
        "activations": acts,
    }
    return bundle


# =============================================================================
# 4. REPORTING HELPERS
# =============================================================================
def print_epoch_table(b):
    print("\n  Epoch |  train_loss |  val_loss |  train_acc |  val_acc")
    print("  ------+-------------+-----------+------------+---------")
    for i in range(len(b["train_loss"])):
        print(f"  {i+1:5d} | {b['train_loss'][i]:11.4f} | "
              f"{b['val_loss'][i]:9.4f} | {b['train_acc'][i]:10.4f} | "
              f"{b['val_acc'][i]:8.4f}")


def print_gradient_table(b):
    g = b["grad_final"]
    if not g:
        return
    print("\n  Final mean |gradient| per layer")
    print("  ---------------------------------")
    for name, val in g.items():
        bar = "#" * min(40, int(-np.log10(val + 1e-30)) * 3) if val > 0 else ""
        print(f"    {name:<10} : {val:.3e}   {bar}")


def print_activation_table(b):
    a = b["activations"]
    if not a:
        return
    print("\n  Neuron activation statistics (probe batch of 512)")
    print("  -------------------------------------------------")
    for name, s in a.items():
        print(f"    {name:<10} : dead {s['dead_fraction']*100:6.2f}% "
              f"of {s['units']:4d} units | mean act {s['mean_activation']:.4f}")


def loss_progress(b):
    """Total drop in training loss across the run."""
    return b["train_loss"][0] - b["train_loss"][-1]


def save_plot(bundles, fname, title):
    if not PLOTTING:
        return
    plt.figure(figsize=(11, 4))
    plt.subplot(1, 2, 1)
    for b in bundles:
        ep = range(1, len(b["train_loss"]) + 1)
        plt.plot(ep, b["train_loss"], label=f"{b['label']} train")
        plt.plot(ep, b["val_loss"], "--", label=f"{b['label']} val")
    plt.title("Loss"); plt.xlabel("epoch"); plt.legend(fontsize=7); plt.grid(alpha=.3)

    plt.subplot(1, 2, 2)
    for b in bundles:
        ep = range(1, len(b["train_acc"]) + 1)
        plt.plot(ep, b["train_acc"], label=f"{b['label']} train")
        plt.plot(ep, b["val_acc"], "--", label=f"{b['label']} val")
    plt.title("Accuracy"); plt.xlabel("epoch"); plt.legend(fontsize=7); plt.grid(alpha=.3)

    plt.suptitle(title); plt.tight_layout()
    plt.savefig(fname, dpi=110); plt.close()
    print(f"\n  [plot saved: {fname}]")


# =============================================================================
# 5. DIAGNOSIS RULES  (decisions are made from the measured numbers)
# =============================================================================
def diagnose_low_lr(b):
    drop = loss_progress(b)
    gain = b["train_acc"][-1] - b["train_acc"][0]
    verdict = ("FAILURE: EXTREMELY LOW LEARNING RATE"
               if drop < 0.05 and gain < 0.05 else "No low-LR failure detected")
    print("\n  DIAGNOSIS")
    print(f"    total training-loss reduction : {drop:.5f}")
    print(f"    total accuracy gain           : {gain:.5f}")
    print(f"    final validation accuracy     : {b['val_acc'][-1]:.4f}")
    print(f"    ==> {verdict}")
    if drop < 0.05:
        print("    Reason : weight updates w <- w - lr*grad are far too small,")
        print("             so the loss curve is almost flat. Training would")
        print("             need orders of magnitude more epochs to converge.")
        print("    Fix    : raise lr (1e-3 .. 1e-1) or switch to Adam.")
    b["verdict"] = verdict
    return verdict


def diagnose_high_lr(b):
    tl = np.array(b["train_loss"])
    diverged = bool(np.isnan(tl).any() or np.isinf(tl).any() or tl[-1] > tl[0])
    oscillation = float(np.mean(np.abs(np.diff(np.nan_to_num(tl, nan=1e3)))))
    verdict = ("FAILURE: EXTREMELY HIGH LEARNING RATE (divergence/oscillation)"
               if diverged or oscillation > 0.5 else "No high-LR failure detected")
    print("\n  DIAGNOSIS")
    print(f"    NaN / Inf in loss            : {bool(np.isnan(tl).any() or np.isinf(tl).any())}")
    print(f"    first -> last training loss  : {tl[0]:.4f} -> {tl[-1]:.4f}")
    print(f"    mean epoch-to-epoch |change| : {oscillation:.4f}")
    print(f"    final validation accuracy    : {b['val_acc'][-1]:.4f} "
          f"(random guess = 0.1000)")
    print(f"    ==> {verdict}")
    if diverged:
        print("    Reason : the step overshoots the minimum each update, the")
        print("             weights explode and the loss diverges to NaN/Inf.")
        print("    Fix    : reduce lr, use gradient clipping or Adam + warmup.")
    b["verdict"] = verdict
    return verdict


def diagnose_vanishing(b):
    g = b["grad_final"]
    names = list(g.keys())
    vals = np.array([g[n] for n in names])
    first, last = vals[0], vals[-1]
    ratio = last / (first + 1e-30)
    vanishing = bool(first < 1e-4 and ratio > 10)

    verdict = ("FAILURE: VANISHING GRADIENT"
               if vanishing else "No vanishing gradient detected")
    print("\n  DIAGNOSIS")
    print(f"    gradient at FIRST hidden layer : {first:.3e}")
    print(f"    gradient at LAST  hidden layer : {last:.3e}")
    print(f"    last/first ratio               : {ratio:.1f}x")
    print(f"    final validation accuracy      : {b['val_acc'][-1]:.4f}")
    print(f"    ==> {verdict}")
    if vanishing:
        print("    Reason : sigmoid derivative <= 0.25, so backpropagating")
        print("             through 6 layers shrinks the signal by >= 0.25^6.")
        print("             Early layers receive almost no gradient and stop")
        print("             learning useful features.")
        print("    Fix    : ReLU activations, He initialisation, BatchNorm,")
        print("             residual connections or a shallower network.")
    b["verdict"] = verdict
    return verdict


def diagnose_dead_relu(b):
    acts = b["activations"]
    worst = max(s["dead_fraction"] for s in acts.values()) if acts else 0.0
    avg = float(np.mean([s["dead_fraction"] for s in acts.values()])) if acts else 0.0
    dead = worst > 0.40
    verdict = ("FAILURE: DEAD ReLU NEURONS"
               if dead else "No significant dead-ReLU problem detected")
    print("\n  DIAGNOSIS")
    print(f"    worst layer dead fraction   : {worst*100:.2f}%")
    print(f"    average dead fraction       : {avg*100:.2f}%")
    print(f"    final validation accuracy   : {b['val_acc'][-1]:.4f}")
    print(f"    ==> {verdict}")
    if dead:
        print("    Reason : a large learning rate drove pre-activations")
        print("             permanently negative; ReLU outputs 0 and its")
        print("             gradient is 0, so those neurons can never recover.")
        print("    Fix    : LeakyReLU / ELU / GELU, smaller lr, He init.")
    b["verdict"] = verdict
    return verdict


def diagnose_overfitting(b):
    gap_acc = b["train_acc"][-1] - b["val_acc"][-1]
    best_val = float(np.min(b["val_loss"]))
    best_ep = int(np.argmin(b["val_loss"])) + 1
    rise = b["val_loss"][-1] - best_val
    over = gap_acc > 0.12 or rise > 0.10
    verdict = "FAILURE: OVERFITTING" if over else "No overfitting detected"
    print("\n  DIAGNOSIS")
    print(f"    final train acc / val acc  : {b['train_acc'][-1]:.4f} / {b['val_acc'][-1]:.4f}")
    print(f"    generalisation gap         : {gap_acc:.4f}")
    print(f"    best val loss              : {best_val:.4f} at epoch {best_ep}")
    print(f"    val-loss rise after best   : {rise:.4f}")
    print(f"    ==> {verdict}")
    if over:
        print("    Reason : the network memorises the small training set;")
        print("             training loss keeps falling while validation loss")
        print(f"             turns upward after epoch {best_ep}.")
        print("    Fix    : dropout, L2 regularisation, early stopping,")
        print("             data augmentation, more training data.")
    b["verdict"] = verdict
    return verdict


def diagnose_stable(b):
    gap = b["train_acc"][-1] - b["val_acc"][-1]
    drop = loss_progress(b)
    g = np.array(list(b["grad_final"].values()))
    dead = max((s["dead_fraction"] for s in b["activations"].values()), default=0)
    healthy = (drop > 0.2 and gap < 0.10 and g.min() > 1e-5 and dead < 0.25
               and b["val_acc"][-1] > 0.80)
    verdict = ("STABLE / HEALTHY CONFIGURATION"
               if healthy else "Configuration is not fully stable")
    print("\n  DIAGNOSIS")
    print(f"    training-loss reduction    : {drop:.4f}")
    print(f"    generalisation gap         : {gap:.4f}")
    print(f"    min layer gradient         : {g.min():.3e}")
    print(f"    worst dead-neuron fraction : {dead*100:.2f}%")
    print(f"    final validation accuracy  : {b['val_acc'][-1]:.4f}")
    print(f"    ==> {verdict}")
    b["verdict"] = verdict
    return verdict


# =============================================================================
# 6. EXPERIMENTS  (menu options 1-6)
# =============================================================================
def exp_low_lr(epochs=8):
    print("\n" + "=" * 70)
    print(" CASE 1 : EXTREMELY LOW LEARNING RATE   (FNN, SGD, lr = 1e-6)")
    print("=" * 70)
    data = load_data()
    m = build_model("FNN", "relu")
    b = train(m, data, lr=1e-6, epochs=epochs, optimizer="sgd",
              label="Low LR (1e-6)")
    print_epoch_table(b); print_gradient_table(b)
    diagnose_low_lr(b)
    save_plot([b], "case1_low_lr.png", "Extremely Low Learning Rate")
    RESULTS["Low LR"] = b
    return b


def exp_high_lr(epochs=8):
    print("\n" + "=" * 70)
    print(" CASE 2 : EXTREMELY HIGH LEARNING RATE  (MLP, SGD, lr = 5.0)")
    print("=" * 70)
    data = load_data()
    m = build_model("MLP", "relu")
    b = train(m, data, lr=5.0, epochs=epochs, optimizer="sgd",
              label="High LR (5.0)")
    print_epoch_table(b); print_gradient_table(b)
    diagnose_high_lr(b)
    save_plot([b], "case2_high_lr.png", "Extremely High Learning Rate")
    RESULTS["High LR"] = b
    return b


def exp_vanishing(epochs=10):
    print("\n" + "=" * 70)
    print(" CASE 3 : VANISHING GRADIENT  (DFNN, 6 sigmoid layers, SGD lr=0.01)")
    print("=" * 70)
    data = load_data()
    m = build_model("DFNN", "sigmoid", init="glorot_uniform")
    b = train(m, data, lr=0.01, epochs=epochs, optimizer="sgd",
              label="Vanishing grad (sigmoid DFNN)")
    print_epoch_table(b); print_gradient_table(b)
    diagnose_vanishing(b)
    save_plot([b], "case3_vanishing_gradient.png", "Vanishing Gradient")
    RESULTS["Vanishing Gradient"] = b
    return b


def exp_dead_relu(epochs=8):
    print("\n" + "=" * 70)
    print(" CASE 4 : DEAD ReLU NEURONS  (MLP, ReLU, SGD lr = 1.0)")
    print("=" * 70)
    data = load_data()
    m = build_model("MLP", "relu", init="he_normal")
    b = train(m, data, lr=1.0, epochs=epochs, optimizer="sgd",
              label="Dead ReLU (lr=1.0)")
    print_epoch_table(b); print_activation_table(b); print_gradient_table(b)
    diagnose_dead_relu(b)
    save_plot([b], "case4_dead_relu.png", "Dead ReLU Neurons")
    RESULTS["Dead ReLU"] = b
    return b


def exp_overfitting(epochs=40):
    print("\n" + "=" * 70)
    print(" CASE 5 : OVERFITTING  (DFNN, 1000 training images, no regulariser)")
    print("=" * 70)
    data = load_data(tiny_train=1000)
    m = build_model("DFNN", "relu", init="he_normal", width=256)
    b = train(m, data, lr=1e-3, epochs=epochs, optimizer="adam",
              batch_size=32, label="Overfitting")
    print_epoch_table(b)
    diagnose_overfitting(b)
    save_plot([b], "case5_overfitting.png", "Overfitting")
    RESULTS["Overfitting"] = b
    return b


def exp_stable(epochs=12):
    print("\n" + "=" * 70)
    print(" CASE 6 : STABLE CONFIGURATION  (MLP, ReLU+He, Adam 1e-3, dropout")
    print("          0.2 + L2 1e-4)")
    print("=" * 70)
    data = load_data()
    m = build_model("MLP", "relu", init="he_normal", dropout=0.2, l2=1e-4,
                    width=256)
    b = train(m, data, lr=1e-3, epochs=epochs, optimizer="adam",
              label="Stable")
    print_epoch_table(b); print_gradient_table(b); print_activation_table(b)
    diagnose_stable(b)
    save_plot([b], "case6_stable.png", "Stable Configuration")
    RESULTS["Stable"] = b
    return b


# =============================================================================
# 7. COMPARISON (menu option 7)
# =============================================================================
def compare_all():
    print("\n" + "=" * 70)
    print(" CASE 7 : RUN AND COMPARE ALL FAILURE CASES")
    print("=" * 70)
    exp_low_lr(); exp_high_lr(); exp_vanishing()
    exp_dead_relu(); exp_overfitting(); exp_stable()

    print("\n\n" + "=" * 104)
    print(" COMPARATIVE SUMMARY".center(104))
    print("=" * 104)
    hdr = (f"{'Case':<20}{'Model':<14}{'Opt':<6}{'LR':<9}"
           f"{'TrAcc':<8}{'ValAcc':<8}{'Gap':<8}{'MinGrad':<11}{'Dead%':<8}")
    print(hdr); print("-" * 104)
    for name, b in RESULTS.items():
        g = list(b["grad_final"].values())
        mg = min(g) if g else float("nan")
        dead = max((s["dead_fraction"] for s in b["activations"].values()),
                   default=0.0) * 100
        gap = b["train_acc"][-1] - b["val_acc"][-1]
        print(f"{name:<20}{b['model']:<14}{b['optimizer']:<6}{b['lr']:<9.0e}"
              f"{b['train_acc'][-1]:<8.4f}{b['val_acc'][-1]:<8.4f}{gap:<8.4f}"
              f"{mg:<11.2e}{dead:<8.2f}")
    print("-" * 104)
    print("\n DETECTED BEHAVIOUR / FAILURE")
    print("-" * 104)
    for name, b in RESULTS.items():
        print(f"  {name:<20} -> {b.get('verdict','(not diagnosed)')}")
    print("=" * 104)

    save_plot(list(RESULTS.values()), "case7_comparison.png",
              "All Failure Cases Compared")


# =============================================================================
# 8. MENU
# =============================================================================
MENU = """
==========================================================
      NEURAL NETWORK FAILURE DIAGNOSIS SYSTEM
              (Fashion-MNIST dataset)
==========================================================
 1. Test Extremely Low Learning Rate
 2. Test Extremely High Learning Rate
 3. Detect Vanishing Gradient
 4. Detect Dead ReLU Neurons
 5. Detect Overfitting
 6. Test Stable Neural Network Configuration
 7. Compare All Failure Cases
 8. Exit
==========================================================
"""


def main():
    actions = {"1": exp_low_lr, "2": exp_high_lr, "3": exp_vanishing,
               "4": exp_dead_relu, "5": exp_overfitting, "6": exp_stable,
               "7": compare_all}
    while True:
        print(MENU)
        choice = input("Enter your choice: ").strip()
        if choice == "8":
            print("\nExiting the Neural Network Failure Diagnosis System.")
            break
        fn = actions.get(choice)
        if fn is None:
            print("\n  Invalid choice. Please enter a number from 1 to 8.")
            continue
        try:
            fn()
        except Exception as e:
            print(f"\n  Error while running experiment: {e}")
        input("\nPress Enter to return to the menu...")


if __name__ == "__main__":
    main()


      NEURAL NETWORK FAILURE DIAGNOSIS SYSTEM
              (Fashion-MNIST dataset)
 1. Test Extremely Low Learning Rate
 2. Test Extremely High Learning Rate
 3. Detect Vanishing Gradient
 4. Detect Dead ReLU Neurons
 5. Detect Overfitting
 6. Test Stable Neural Network Configuration
 7. Compare All Failure Cases
 8. Exit

Enter your choice: 1

 CASE 1 : EXTREMELY LOW LEARNING RATE   (FNN, SGD, lr = 1e-6)
29515/29515 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
26421880/26421880 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
5148/5148 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
4422102/4422102 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step

  >> Training FNN_relu | optimizer=sgd | lr=1e-06 | epochs=8

  Epoch |  train_loss |  val_loss |  train_acc |  val_acc
  ------+-------------+-----------+------------+---------
      1 |      2.3224 |    2.3180 |     0.1541 |   0.1617
      2 |      2.3222 |    2.3178 |     0.1541 |   0.1620
      3 |      2.3220 |    2.3176 |     0.1542 |   0.1620
      4 |      2.3218 |    2.3174 |     0.1542 |